In [4]:
from google.colab import files

uploaded = files.upload()

Saving car_purchase.csv to car_purchase.csv


In [1]:
import numpy as np
import pandas as pd

# --- Step 1: Load Data set ---
try:
    data = pd.read_csv('car_purchase.csv')
    concepts = np.array(data.iloc[:, 0:-1])
    target = np.array(data.iloc[:, -1])
    print("Step 1: Data Loaded Successfully")
    print(data)
    print("-" * 50)
except FileNotFoundError:
    print("Error: File not found.")
    exit()

def learn(concepts, target):
    # --- Step 2: Initialize General Hypothesis and Specific Hypothesis ---
    # Specific Hypothesis (S) starts as the first positive example
    specific_h = concepts[0].copy()

    # General Hypothesis (G) starts as the most general (all '?')
    # We create a list of generic hypotheses, one for each feature
    general_h = [["?" for i in range(len(specific_h))] for i in range(len(specific_h))]

    print(f"Step 2 Initialization:\nSpecific (S): {specific_h}\nGeneral (G): {general_h}\n")
    print("-" * 50)

    # --- Step 3: For each training example ---
    for i, h in enumerate(concepts):

        # --- Step 4: If example is positive example ---
        if target[i] == "Yes":
            print(f"Instance {i+1} is Positive (Yes)")

            for x in range(len(specific_h)):
                # Logic from image: "if attribute_value != hypothesis_value: replace with '?'"
                if h[x] != specific_h[x]:
                    specific_h[x] = '?'
                    # (Standard requirement: If S generalizes, G must also be updated to remain consistent)
                    general_h[x][x] = '?'

        # --- Step 5: If example is Negative example ---
        if target[i] == "No":
            print(f"Instance {i+1} is Negative (No)")

            # Logic from image: "Make generalize hypothesis more specific"
            for x in range(len(specific_h)):
                # We specialize G to exclude this negative example.
                # We enforce the constraint from S only if the negative example violates it.
                if h[x] != specific_h[x]:
                    general_h[x][x] = specific_h[x]
                else:
                    general_h[x][x] = '?'

        # Display progress after each step
        print(f"Current S: {specific_h}")
        print(f"Current G: {general_h}")
        print("-" * 50)

    # Final cleanup to remove useless general hypotheses (those that are still all '?')
    # Improved cleanup to avoid modifying list during iteration
    num_attributes = len(specific_h)
    all_question_marks = ['?' for _ in range(num_attributes)]
    general_h = [g for g in general_h if g != all_question_marks]

    return specific_h, general_h

# --- Prediction Function ---
def predict(h, s_final, g_final):
    # 1. Check against Specific Boundary (S)
    # If S has a specific value (not '?'), the new instance MUST match it.
    for i in range(len(s_final)):
        if s_final[i] != '?' and s_final[i] != h[i]:
            return "No (Violates Specific Hypothesis)"

    # 2. Check against General Boundary (G)
    # The instance must satisfy at least one of the general hypotheses (usually all in a consistent space)
    # In this simple implementation, we strictly check if it violates known constraints.
    for i in range(len(g_final)):
        matching_g = True
        for j in range(len(g_final[i])):
            if g_final[i][j] != '?' and g_final[i][j] != h[j]:
                matching_g = False
                break # Break inner loop if a mismatch is found
        if not matching_g:
             return "No (Violates General Hypothesis)"

    return "Yes"

# Run the algorithm
s_final, g_final = learn(concepts, target)

print("\nFinal Specific Hypothesis:", s_final)
print("Final General Hypothesis:", g_final)

print("\nTraining Completed.")
print(f"Final S: {s_final}")
print(f"Final G: {g_final}\n")
print("-" * 50)

# 2. Test Cases (Prediction)
# Let's create some new cars to test
test_data = [
    ['Red', 'Sports', 'Domestic', '2'],
    ['Green', 'Sports', 'Domestic', '4'],
    ['Red', 'Economy', 'Domestic', '4'],
    ['Red', 'Sports', 'Imported', '2']
]

print("Prediction Results:\n")
# 'columns' variable is not used in the loop, kept for context from prior version
# columns = ['Color', 'Type', 'Origin', 'Doors']

for test_instance in test_data:
    result = predict(test_instance, s_final, g_final)
    print(f"Test Instance: {test_instance}")
    print(f"Prediction:    {result}")
    print("-" * 20)

Step 1: Data Loaded Successfully
    Color     Type    Origin  Doors Buy (Target)
0     Red   Sports  Domestic      2          Yes
1     Red   Sports  Domestic      4          Yes
2     Red  Economy  Domestic      4           No
3  Yellow   Sports  Domestic      2          Yes
--------------------------------------------------
Step 2 Initialization:
Specific (S): ['Red' 'Sports' 'Domestic' 2]
General (G): [['?', '?', '?', '?'], ['?', '?', '?', '?'], ['?', '?', '?', '?'], ['?', '?', '?', '?']]

--------------------------------------------------
Instance 1 is Positive (Yes)
Current S: ['Red' 'Sports' 'Domestic' 2]
Current G: [['?', '?', '?', '?'], ['?', '?', '?', '?'], ['?', '?', '?', '?'], ['?', '?', '?', '?']]
--------------------------------------------------
Instance 2 is Positive (Yes)
Current S: ['Red' 'Sports' 'Domestic' '?']
Current G: [['?', '?', '?', '?'], ['?', '?', '?', '?'], ['?', '?', '?', '?'], ['?', '?', '?', '?']]
--------------------------------------------------
Instan